# Aula 04 — Laboratório
## Muitas variáveis, poucas que importam

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Dois conjuntos de dados, o mesmo roteiro:

1. o **mundo simulado** da Aula 03, agora com 40 perguntas do questionário que não
   afetam o salário;
2. **vendas reais de imóveis** em Ames, Iowa, de 2006 a 2010, com 74 características
   por venda (De Cock, 2011).

Em cada um: OLS com tudo, Ridge e Lasso; a penalização escolhida por validação cruzada;
a lista de variáveis que o Lasso mantém — e quanto confiar nela.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import io, os, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")          # avisos de convergencia do Lasso
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Laranja = OLS / fora da amostra.
# Azul = Ridge / dentro da amostra. Verde = Lasso. Cinza = coluna inutil.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#8A8D96"

try:
    import sklearn
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV
    from sklearn.model_selection import KFold
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")


def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))


print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("\nAmbiente pronto.")

## Bloco 1 — aquecimento com a verdade na mão

150 trabalhadores e 52 colunas: 12 que entram no salário e 40 que não entram.

In [ ]:
# Bloco 1 — o mundo simulado da Aula 03, com o questionario inteiro.
#
# As 12 primeiras colunas sao as de sempre. As colunas q01 a q40 sao perguntas
# do questionario que NAO entram no salario; q01 a q10 andam junto com estudo,
# experiencia ou origem familiar. Aqui a verdade e conhecida: da para conferir
# o que o Lasso mantem e o que ele descarta.

EFEITO_DIRETO = 0.06
EFEITO_HABILIDADE = 0.26
EFEITO_EXPERIENCIA = 0.03
EFEITO_EXPERIENCIA2 = -0.0005
EFEITO_CARGO = 0.8
RUIDO = 0.33
CORR_ORIGEM_HAB = 0.92
SETORES = ('Serviços', 'Indústria', 'Comércio', 'Público')
EFEITO_SETOR = {'Serviços': 0.0, 'Indústria': 0.1, 'Comércio': -0.08, 'Público': 0.18}
REGIOES = ('Nordeste', 'Sudeste', 'Sul', 'Norte', 'Centro-Oeste')
EFEITO_REGIAO = {'Nordeste': 0.0, 'Sudeste': 0.2, 'Sul': 0.14, 'Norte': 0.02, 'Centro-Oeste': 0.1}
N_IRRELEVANTES = 40
N_CORRELACIONADAS = 10
N_TREINO_SIM = 150
RELEVANTES = ['estudo', 'experiencia', 'experiencia2', 'origem', 'cargo', 'setor_Indústria', 'setor_Comércio', 'setor_Público', 'regiao_Sudeste', 'regiao_Sul', 'regiao_Norte', 'regiao_Centro-Oeste']


def gerar_base(n=4000, semente=42, anos_a_mais=0.0):
    """Uma pesquisa domiciliar fictícia de rendimentos.

    Estrutura de uma PNAD reduzida: uma linha por trabalhador, salário mensal,
    anos de estudo, experiência, origem familiar, setor, região e ocupação de
    chefia. A HABILIDADE existe no mundo e entra no salário, mas NÃO é
    observada — é ela que faz a regressão ingênua mentir.

    `anos_a_mais` faz a intervenção: dá a todo mundo esse tanto de estudo a
    mais, mantendo habilidade, sorteios e ruídos idênticos. É assim que o
    efeito causal verdadeiro é medido, e não lido de um parâmetro.
    """
    rng = np.random.default_rng(semente)

    # habilidade: existe, afeta estudo e salário, e ninguém mede
    hab = rng.normal(0, 1, n)

    # origem familiar (escolaridade dos pais, padronizada): observada, e
    # correlacionada com a habilidade — por isso serve de proxy imperfeita
    origem = (CORR_ORIGEM_HAB * hab
              + np.sqrt(1 - CORR_ORIGEM_HAB ** 2) * rng.normal(0, 1, n))

    # anos de estudo: quem tem mais habilidade e melhor origem estuda mais
    estudo = 9.0 + 1.00 * origem + 1.00 * hab + rng.normal(0, 1.8, n)
    estudo = np.clip(np.round(estudo), 4, 20) + anos_a_mais

    # experiência: anos de trabalho, independente do estudo neste mundo
    experiencia = np.clip(np.round(rng.normal(18, 9, n)), 0, 40)

    setor = rng.choice(SETORES, n, p=[0.45, 0.20, 0.22, 0.13])
    regiao = rng.choice(REGIOES, n, p=[0.30, 0.34, 0.14, 0.08, 0.14])

    # cargo de chefia: CONSEQUÊNCIA do estudo e da habilidade. Controlar por
    # ele fecha parte do caminho pelo qual o estudo aumenta o salário.
    indice = 0.85 * (estudo - 11) / 3 + 0.70 * hab + 0.03 * experiencia
    cargo = (indice + rng.normal(0, 1, n) > 1.10).astype(int)

    log_salario = (
        6.40
        + EFEITO_DIRETO * estudo
        + EFEITO_EXPERIENCIA * experiencia
        + EFEITO_EXPERIENCIA2 * experiencia ** 2
        + EFEITO_HABILIDADE * hab
        + EFEITO_CARGO * cargo
        + np.array([EFEITO_SETOR[s] for s in setor])
        + np.array([EFEITO_REGIAO[r] for r in regiao])
        + rng.normal(0, RUIDO, n)
    )

    return {
        "salario": np.round(np.exp(log_salario), 2),
        "log_salario": log_salario,
        "estudo": estudo,
        "experiencia": experiencia,
        "origem": np.round(origem, 3),
        "setor": setor,
        "regiao": regiao,
        "cargo": cargo,
        "_habilidade": hab,          # não entregar ao aluno
    }

def dummies(valores, categorias):
    """Uma coluna 0/1 por categoria, menos a primeira (a de referência)."""
    return np.column_stack([(valores == c).astype(float) for c in categorias[1:]])


def gerar_cadastro(n=N_TREINO_SIM, semente=42, n_irrelevantes=N_IRRELEVANTES,
                   n_correlacionadas=N_CORRELACIONADAS):
    """A pesquisa de rendimentos da Aula 03, com o questionário inteiro.

    As 12 primeiras colunas são as de sempre (estudo, experiência e seu
    quadrado, origem familiar, cargo de chefia, setor e região). As colunas
    q01, q02, … são perguntas do questionário que NÃO entram no salário. As
    `n_correlacionadas` primeiras andam junto com estudo, experiência ou
    origem — parecem informativas, mas não são.
    """
    b = gerar_base(n=n, semente=semente)
    rng = np.random.default_rng(semente + 7)
    exp = b["experiencia"]
    X = np.column_stack([
        b["estudo"], exp, exp ** 2 / 100, b["origem"], b["cargo"],
        dummies(b["setor"], SETORES), dummies(b["regiao"], REGIOES),
    ])
    padrao = np.column_stack([(b["estudo"] - 11) / 3, (exp - 18) / 9, b["origem"]])
    J = rng.normal(0, 1, (n, n_irrelevantes))
    for j in range(n_correlacionadas):
        J[:, j] = 0.7 * padrao[:, j % 3] + np.sqrt(1 - 0.7 ** 2) * J[:, j]
    nomes = RELEVANTES + [f"q{j:02d}" for j in range(1, n_irrelevantes + 1)]
    return pd.DataFrame(np.column_stack([X, J]), columns=nomes), b["log_salario"]


X_sim, y_sim = gerar_cadastro(N_TREINO_SIM, 42)          # 150 no treino
X_sim_teste, y_sim_teste = gerar_cadastro(4000, 999)     # 4.000 que o modelo nao vera
print(f"treino: {X_sim.shape[0]} pessoas x {X_sim.shape[1]} colunas")
print(f"teste : {X_sim_teste.shape[0]} pessoas")
X_sim.head()

In [ ]:
# Bloco 1 — o OLS com todas as colunas. COMPLETE.
#
# Ajuste um pipeline que padroniza as colunas e roda a regressao linear.
# Meca o erro (RMSE) dentro da amostra e fora dela.

# COMPLETE
ols_sim = ___

print(f"OLS  dentro da amostra: {___:.4f}")
print(f"OLS  fora da amostra  : {___:.4f}")

In [ ]:
# Bloco 1 — o Lasso, com lambda escolhido por validacao cruzada. COMPLETE.
#
# No scikit-learn o lambda se chama `alpha`. LassoCV testa cada valor da grade
# em 5 dobras e fica com o de menor erro medio.

ALPHAS_SIM = np.logspace(-3.5, 0, 50)

# COMPLETE: padronize e rode o LassoCV com a grade acima, 5 dobras embaralhadas
# (KFold(5, shuffle=True, random_state=0)) e max_iter=50000
lasso_sim = make_pipeline(
    ___,
    LassoCV(alphas=___, cv=___, max_iter=50000),
).fit(X_sim, y_sim)

print(f"lambda escolhido       : {lasso_sim[-1].alpha_:.4f}")
print(f"Lasso dentro da amostra: {rmse(y_sim, lasso_sim.predict(X_sim)):.4f}")
print(f"Lasso fora da amostra  : {___:.4f}")

In [ ]:
# Bloco 1 — quem sobreviveu? COMPLETE.
coef_sim = pd.Series(lasso_sim[-1].coef_, index=X_sim.columns)

# COMPLETE: liste as colunas RELEVANTES que ficaram e as que sairam
relevantes_mantidas = ___
relevantes_fora = ___
# COMPLETE: quantas das 40 perguntas inuteis (q01 a q40) ficaram?
inuteis_mantidas = ___

print("relevantes mantidas:", relevantes_mantidas)
print("relevantes fora    :", relevantes_fora)
print("inuteis mantidas   :", len(inuteis_mantidas), "de", N_IRRELEVANTES, inuteis_mantidas)
print("o cargo de chefia ficou?", coef_sim["cargo"] != 0)

## Bloco 2 — Ames: da tabela crua à matriz do modelo

Cada linha é uma venda de imóvel residencial em Ames, Iowa, entre 2006 e 2010. O preço
está em dólares; as áreas, em pés quadrados (1 m² ≈ 10,76 pés²).

In [ ]:
# Bloco 2 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (niveis e frequencias das categorias; media, desvio,
# limites e fracao de zeros das colunas numericas) e uma equacao simples de
# preco. Serve so para gerar vendas inventadas com a mesma estrutura.

ESP_AMES = {'ordem': ['MS_SubClass', 'MS_Zoning', 'Lot_Frontage', 'Lot_Area', 'Street', 'Alley', 'Lot_Shape',
           'Land_Contour', 'Utilities', 'Lot_Config', 'Land_Slope', 'Neighborhood', 'Condition_1',
           'Condition_2', 'Bldg_Type', 'House_Style', 'Overall_Cond', 'Year_Built',
           'Year_Remod_Add', 'Roof_Style', 'Roof_Matl', 'Exterior_1st', 'Exterior_2nd',
           'Mas_Vnr_Type', 'Mas_Vnr_Area', 'Exter_Cond', 'Foundation', 'Bsmt_Cond', 'Bsmt_Exposure',
           'BsmtFin_Type_1', 'BsmtFin_SF_1', 'BsmtFin_Type_2', 'BsmtFin_SF_2', 'Bsmt_Unf_SF',
           'Total_Bsmt_SF', 'Heating', 'Heating_QC', 'Central_Air', 'Electrical', 'First_Flr_SF',
           'Second_Flr_SF', 'Gr_Liv_Area', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath',
           'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Functional',
           'Fireplaces', 'Garage_Type', 'Garage_Finish', 'Garage_Cars', 'Garage_Area',
           'Garage_Cond', 'Paved_Drive', 'Wood_Deck_SF', 'Open_Porch_SF', 'Enclosed_Porch',
           'Three_season_porch', 'Screen_Porch', 'Pool_Area', 'Pool_QC', 'Fence', 'Misc_Feature',
           'Misc_Val', 'Mo_Sold', 'Year_Sold', 'Sale_Type', 'Sale_Condition', 'Sale_Price',
           'Longitude', 'Latitude'],
 'cat': {'MS_SubClass': [['One_Story_1946_and_Newer_All_Styles', 'Two_Story_1946_and_Newer',
                          'One_and_Half_Story_Finished_All_Ages', 'One_Story_PUD_1946_and_Newer',
                          'One_Story_1945_and_Older', 'Two_Story_PUD_1946_and_Newer',
                          'Two_Story_1945_and_Older', 'Split_or_Multilevel',
                          'Duplex_All_Styles_and_Ages', 'Two_Family_conversion_All_Styles_and_Ages',
                          'Split_Foyer', 'Two_and_Half_Story_All_Ages',
                          'One_and_Half_Story_Unfinished_All_Ages',
                          'PUD_Multilevel_Split_Level_Foyer',
                          'One_Story_with_Finished_Attic_All_Ages',
                          'One_and_Half_Story_PUD_All_Ages'],
                         [0.3683, 0.1962, 0.098, 0.0655, 0.0474, 0.044, 0.0437, 0.0403, 0.0372,
                          0.0208, 0.0164, 0.0078, 0.0061, 0.0058, 0.002, 0.0003]],
         'MS_Zoning': [['Residential_Low_Density', 'Residential_Medium_Density',
                        'Floating_Village_Residential', 'Residential_High_Density', 'C_all',
                        'I_all', 'A_agr'],
                       [0.7758, 0.1577, 0.0474, 0.0092, 0.0085, 0.0007, 0.0007]],
         'Street': [['Pave', 'Grvl'], [0.9959, 0.0041]],
         'Alley': [['No_Alley_Access', 'Gravel', 'Paved'], [0.9324, 0.041, 0.0266]],
         'Lot_Shape': [['Regular', 'Slightly_Irregular', 'Moderately_Irregular', 'Irregular'],
                       [0.6345, 0.3341, 0.0259, 0.0055]],
         'Land_Contour': [['Lvl', 'HLS', 'Bnk', 'Low'], [0.8986, 0.041, 0.0399, 0.0205]],
         'Utilities': [['AllPub', 'NoSewr', 'NoSeWa'], [0.999, 0.0007, 0.0003]],
         'Lot_Config': [['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3'],
                        [0.7304, 0.1744, 0.0614, 0.029, 0.0048]],
         'Land_Slope': [['Gtl', 'Mod', 'Sev'], [0.9519, 0.0427, 0.0055]],
         'Neighborhood': [['North_Ames', 'College_Creek', 'Old_Town', 'Edwards', 'Somerset',
                           'Northridge_Heights', 'Gilbert', 'Sawyer', 'Northwest_Ames',
                           'Sawyer_West', 'Mitchell', 'Brookside', 'Crawford',
                           'Iowa_DOT_and_Rail_Road', 'Timberland', 'Northridge', 'Stone_Brook',
                           'South_and_West_of_Iowa_State_University', 'Clear_Creek',
                           'Meadow_Village', 'Briardale', 'Bloomington_Heights', 'Veenker',
                           'Northpark_Villa', 'Blueste', 'Greens', 'Green_Hills', 'Landmark'],
                          [0.1512, 0.0911, 0.0816, 0.0662, 0.0621, 0.0567, 0.0563, 0.0515, 0.0447,
                           0.0427, 0.0389, 0.0369, 0.0352, 0.0317, 0.0246, 0.0242, 0.0174, 0.0164,
                           0.015, 0.0126, 0.0102, 0.0096, 0.0082, 0.0078, 0.0034, 0.0027, 0.0007,
                           0.0003]],
         'Condition_1': [['Norm', 'Feedr', 'Artery', 'RRAn', 'PosN', 'RRAe', 'PosA', 'RRNn',
                          'RRNe'],
                         [0.8608, 0.056, 0.0314, 0.0171, 0.0133, 0.0096, 0.0068, 0.0031, 0.002]],
         'Condition_2': [['Norm', 'Feedr', 'Artery', 'PosA', 'PosN', 'RRNn', 'RRAe', 'RRAn'],
                         [0.9898, 0.0044, 0.0017, 0.0014, 0.0014, 0.0007, 0.0003, 0.0003]],
         'Bldg_Type': [['OneFam', 'TwnhsE', 'Duplex', 'Twnhs', 'TwoFmCon'],
                       [0.8276, 0.0795, 0.0372, 0.0345, 0.0212]],
         'House_Style': [['One_Story', 'Two_Story', 'One_and_Half_Fin', 'SLvl', 'SFoyer',
                          'Two_and_Half_Unf', 'One_and_Half_Unf', 'Two_and_Half_Fin'],
                         [0.5055, 0.298, 0.1072, 0.0437, 0.0283, 0.0082, 0.0065, 0.0027]],
         'Overall_Cond': [['Average', 'Above_Average', 'Good', 'Very_Good', 'Below_Average', 'Fair',
                           'Excellent', 'Poor', 'Very_Poor'],
                          [0.5645, 0.1819, 0.1331, 0.0491, 0.0345, 0.0171, 0.014, 0.0034, 0.0024]],
         'Roof_Style': [['Gable', 'Hip', 'Gambrel', 'Flat', 'Mansard', 'Shed'],
                        [0.7922, 0.1881, 0.0075, 0.0068, 0.0038, 0.0017]],
         'Roof_Matl': [['CompShg', 'Tar&Grv', 'WdShake', 'WdShngl', 'Membran', 'ClyTile', 'Roll',
                        'Metal'],
                       [0.9853, 0.0078, 0.0031, 0.0024, 0.0003, 0.0003, 0.0003, 0.0003]],
         'Exterior_1st': [['VinylSd', 'MetalSd', 'HdBoard', 'Wd Sdng', 'Plywood', 'CemntBd',
                           'BrkFace', 'WdShing', 'AsbShng', 'Stucco', 'BrkComm', 'AsphShn',
                           'CBlock', 'Stone', 'PreCast', 'ImStucc'],
                          [0.3502, 0.1536, 0.1509, 0.1433, 0.0754, 0.043, 0.03, 0.0191, 0.015,
                           0.0147, 0.002, 0.0007, 0.0007, 0.0007, 0.0003, 0.0003]],
         'Exterior_2nd': [['VinylSd', 'MetalSd', 'HdBoard', 'Wd Sdng', 'Plywood', 'CmentBd',
                           'Wd Shng', 'BrkFace', 'Stucco', 'AsbShng', 'Brk Cmn', 'ImStucc', 'Stone',
                           'AsphShn', 'CBlock', 'PreCast', 'Other'],
                          [0.3464, 0.1526, 0.1386, 0.1355, 0.0935, 0.043, 0.0276, 0.016, 0.016,
                           0.013, 0.0075, 0.0051, 0.002, 0.0014, 0.001, 0.0003, 0.0003]],
         'Mas_Vnr_Type': [['', 'BrkFace', 'Stone', 'BrkCmn', 'CBlock'],
                          [0.6058, 0.3003, 0.085, 0.0085, 0.0003]],
         'Exter_Cond': [['Typical', 'Good', 'Fair', 'Excellent', 'Poor'],
                        [0.87, 0.102, 0.0229, 0.0041, 0.001]],
         'Foundation': [['PConc', 'CBlock', 'BrkTil', 'Slab', 'Stone', 'Wood'],
                        [0.4471, 0.4246, 0.1061, 0.0167, 0.0038, 0.0017]],
         'Bsmt_Cond': [['Typical', 'Good', 'Fair', 'No_Basement', 'Poor', 'Excellent'],
                       [0.8928, 0.0416, 0.0355, 0.0273, 0.0017, 0.001]],
         'Bsmt_Exposure': [['No', 'Av', 'Gd', 'Mn', 'No_Basement'],
                           [0.6505, 0.1427, 0.0969, 0.0816, 0.0283]],
         'BsmtFin_Type_1': [['GLQ', 'Unf', 'ALQ', 'Rec', 'BLQ', 'LwQ', 'No_Basement'],
                            [0.2932, 0.2904, 0.1464, 0.0983, 0.0918, 0.0526, 0.0273]],
         'BsmtFin_Type_2': [['Unf', 'Rec', 'LwQ', 'No_Basement', 'BLQ', 'ALQ', 'GLQ'],
                            [0.8529, 0.0362, 0.0304, 0.0276, 0.0232, 0.0181, 0.0116]],
         'Heating': [['GasA', 'GasW', 'Grav', 'Wall', 'OthW', 'Floor'],
                     [0.9846, 0.0092, 0.0031, 0.002, 0.0007, 0.0003]],
         'Heating_QC': [['Excellent', 'Typical', 'Good', 'Fair', 'Poor'],
                        [0.5102, 0.2949, 0.1625, 0.0314, 0.001]],
         'Central_Air': [['Y', 'N'], [0.9331, 0.0669]],
         'Electrical': [['SBrkr', 'FuseA', 'FuseF', 'FuseP', 'Unknown', 'Mix'],
                        [0.9154, 0.0642, 0.0171, 0.0027, 0.0003, 0.0003]],
         'Functional': [['Typ', 'Min2', 'Min1', 'Mod', 'Maj1', 'Maj2', 'Sev', 'Sal'],
                        [0.9311, 0.0239, 0.0222, 0.0119, 0.0065, 0.0031, 0.0007, 0.0007]],
         'Garage_Type': [['Attchd', 'Detchd', 'BuiltIn', 'No_Garage', 'Basment',
                          'More_Than_Two_Types', 'CarPort'],
                         [0.5908, 0.2669, 0.0635, 0.0536, 0.0123, 0.0078, 0.0051]],
         'Garage_Finish': [['Unf', 'RFn', 'Fin', 'No_Garage'], [0.4201, 0.2771, 0.2485, 0.0543]],
         'Garage_Cond': [['Typical', 'No_Garage', 'Fair', 'Good', 'Poor', 'Excellent'],
                         [0.9096, 0.0543, 0.0253, 0.0051, 0.0048, 0.001]],
         'Paved_Drive': [['Paved', 'Dirt_Gravel', 'Partial_Pavement'], [0.9051, 0.0737, 0.0212]],
         'Pool_QC': [['No_Pool', 'Excellent', 'Good', 'Typical', 'Fair'],
                     [0.9956, 0.0014, 0.0014, 0.001, 0.0007]],
         'Fence': [['No_Fence', 'Minimum_Privacy', 'Good_Privacy', 'Good_Wood',
                    'Minimum_Wood_Wire'],
                   [0.8048, 0.1126, 0.0403, 0.0382, 0.0041]],
         'Misc_Feature': [['', 'Shed', 'Gar2', 'Othr', 'Elev', 'TenC'],
                          [0.9638, 0.0324, 0.0017, 0.0014, 0.0003, 0.0003]],
         'Sale_Type': [['WD ', 'New', 'COD', 'ConLD', 'CWD', 'ConLI', 'ConLw', 'Oth', 'Con', 'VWD'],
                       [0.8655, 0.0816, 0.0297, 0.0089, 0.0041, 0.0031, 0.0027, 0.0024, 0.0017,
                        0.0003]],
         'Sale_Condition': [['Normal', 'Partial', 'Abnorml', 'Family', 'Alloca', 'AdjLand'],
                            [0.8235, 0.0836, 0.0648, 0.0157, 0.0082, 0.0041]]},
 'disc': {'BsmtFin_SF_1': [[0, 1, 2, 3, 4, 5, 6, 7],
                           [0.0003, 0.1464, 0.0918, 0.2932, 0.0526, 0.027, 0.0983, 0.2904]],
          'Bsmt_Full_Bath': [[0, 1, 2, 3], [0.5833, 0.4031, 0.013, 0.0007]],
          'Bsmt_Half_Bath': [[0, 1, 2], [0.9403, 0.0584, 0.0014]],
          'Full_Bath': [[0, 1, 2, 3, 4], [0.0041, 0.4498, 0.5229, 0.0218, 0.0014]],
          'Half_Bath': [[0, 1, 2], [0.629, 0.3625, 0.0085]],
          'Bedroom_AbvGr': [[0, 1, 2, 3, 4, 5, 6, 8],
                            [0.0027, 0.0382, 0.2536, 0.5451, 0.1365, 0.0164, 0.0072, 0.0003]],
          'Kitchen_AbvGr': [[0, 1, 2, 3], [0.001, 0.9543, 0.044, 0.0007]],
          'Fireplaces': [[0, 1, 2, 3, 4], [0.4853, 0.4348, 0.0754, 0.0041, 0.0003]],
          'Garage_Cars': [[0, 1, 2, 3, 4, 5], [0.0539, 0.2655, 0.5471, 0.1276, 0.0055, 0.0003]],
          'Mo_Sold': [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
                      [0.042, 0.0454, 0.0792, 0.0952, 0.1348, 0.1724, 0.1532, 0.0795, 0.0549, 0.059,
                       0.0488, 0.0355]],
          'Year_Sold': [[2006, 2007, 2008, 2009, 2010], [0.2133, 0.2369, 0.2123, 0.2212, 0.1164]]},
 'num': {'Lot_Frontage': [69.2246, 23.3653, 0.0, 313.0, 1, 0.1672, 0.187, 0.026],
         'Lot_Area': [10147.9218, 7880.0178, 1300.0, 215245.0, 1, 0.0, 0.286, 0.023],
         'Year_Built': [1971.3563, 30.2454, 1872.0, 2010.0, 1, 0.0, 0.242, 1.0],
         'Year_Remod_Add': [1984.2666, 20.8603, 1950.0, 2010.0, 1, 0.0, 0.317, 0.612],
         'Mas_Vnr_Area': [255.5772, 202.9703, 0.0, 1600.0, 1, 0.6044, 0.401, 0.307],
         'BsmtFin_SF_2': [414.9202, 295.7176, 0.0, 1526.0, 1, 0.8802, -0.018, -0.027],
         'Bsmt_Unf_SF': [610.0857, 423.9064, 0.0, 2336.0, 1, 0.0836, 0.236, 0.129],
         'Total_Bsmt_SF': [1080.7646, 409.8898, 0.0, 6110.0, 1, 0.0273, 0.445, 0.408],
         'First_Flr_SF': [1159.5577, 391.8909, 334.0, 5095.0, 1, 0.0, 0.562, 0.31],
         'Second_Flr_SF': [785.0527, 276.492, 0.0, 2065.0, 1, 0.5727, 0.655, 0.017],
         'Gr_Liv_Area': [1499.6904, 505.5089, 334.0, 5642.0, 1, 0.0, 1.0, 0.242],
         'TotRms_AbvGrd': [6.443, 1.573, 2.0, 15.0, 1, 0.0, 0.808, 0.112],
         'Garage_Area': [499.5992, 188.3649, 0.0, 1488.0, 1, 0.0539, 0.484, 0.481],
         'Wood_Deck_SF': [195.6503, 115.6878, 0.0, 1424.0, 1, 0.5208, 0.25, 0.229],
         'Open_Porch_SF': [85.4436, 70.3357, 0.0, 742.0, 1, 0.4437, 0.341, 0.198],
         'Enclosed_Porch': [146.8932, 89.8419, 0.0, 1012.0, 1, 0.8433, 0.004, -0.374],
         'Three_season_porch': [205.2973, 93.0406, 0.0, 508.0, 1, 0.9874, 0.006, 0.016],
         'Screen_Porch': [183.1484, 73.4901, 0.0, 576.0, 1, 0.9126, 0.087, -0.041],
         'Pool_Area': [505.6154, 183.2223, 0.0, 800.0, 1, 0.9956, 0.135, 0.002],
         'Misc_Val': [1440.3981, 2681.231, 0.0, 17000.0, 1, 0.9648, 0.067, -0.011],
         'Longitude': [-93.6429, 0.0257, -93.6932, -93.5774, 0, 0.0, -0.143, -0.426],
         'Latitude': [42.0345, 0.0184, 41.9865, 42.0634, 0, 0.0, 0.18, 0.251]},
 'preco': {'const': 12.0424,
           'inclinacoes': {'Gr_Liv_Area': 0.1542,
                           'Year_Built': 0.0913,
                           'Total_Bsmt_SF': 0.0705,
                           'Garage_Cars': 0.062},
           'bairro': {'Bloomington_Heights': -0.0224,
                      'Blueste': -0.0813,
                      'Briardale': -0.2361,
                      'Brookside': -0.0601,
                      'Clear_Creek': 0.0912,
                      'College_Creek': 0.0188,
                      'Crawford': 0.1731,
                      'Edwards': -0.1501,
                      'Gilbert': -0.0125,
                      'Green_Hills': 0.5629,
                      'Greens': 0.175,
                      'Iowa_DOT_and_Rail_Road': -0.2314,
                      'Landmark': -0.1769,
                      'Meadow_Village': -0.3126,
                      'Mitchell': -0.0662,
                      'North_Ames': -0.0566,
                      'Northpark_Villa': -0.1116,
                      'Northridge': 0.1238,
                      'Northridge_Heights': 0.1916,
                      'Northwest_Ames': -0.026,
                      'Old_Town': -0.1272,
                      'Sawyer': -0.0716,
                      'Sawyer_West': -0.0485,
                      'Somerset': 0.0807,
                      'South_and_West_of_Iowa_State_University': -0.0763,
                      'Stone_Brook': 0.2234,
                      'Timberland': 0.0993,
                      'Veenker': 0.1275},
           'ruido': 0.1783,
           'medias': {'Gr_Liv_Area': [1499.69, 505.509],
                      'Year_Built': [1971.356, 30.245],
                      'Total_Bsmt_SF': [1051.256, 440.968],
                      'Garage_Cars': [1.766, 0.761]}}}


def ames_sintetica(ESP, n=2930, semente=2024):
    """Plano B: vendas INVENTADAS com as mesmas colunas, os mesmos tipos e os
    mesmos níveis da base real. Área e idade puxam as demais colunas numéricas;
    o preço sai de área, idade, porão, garagem e bairro, mais ruído."""
    rng = np.random.default_rng(semente)
    tam = rng.normal(0, 1, n)          # fator comum de tamanho
    ida = rng.normal(0, 1, n)          # fator comum de idade
    d = {}
    for c, (media, dp, lo, hi, inteiro, f0, r_tam, r_ida) in ESP["num"].items():
        if c == "Gr_Liv_Area":
            zc = tam
        elif c == "Year_Built":
            zc = ida
        else:
            a, b = r_tam, r_ida
            resto = max(0.0, 1 - a * a - b * b)
            zc = a * tam + b * ida + np.sqrt(resto) * rng.normal(0, 1, n)
        v = np.clip(media + dp * zc, lo, hi)
        v = np.where(rng.random(n) < f0, 0, v)
        d[c] = np.round(v).astype(int) if inteiro else v
    for c, (valores, probs) in ESP["disc"].items():
        p = np.array(probs, dtype=float)
        d[c] = rng.choice(np.array(valores), n, p=p / p.sum())
    for c, (niveis, probs) in ESP["cat"].items():
        p = np.array(probs, dtype=float)
        v = rng.choice(np.array(niveis, dtype=object), n, p=p / p.sum())
        d[c] = np.where(v == "", None, v)
    P = ESP["preco"]
    lp = np.full(n, P["const"])
    for c, b in P["inclinacoes"].items():
        m, s = P["medias"][c]
        lp += b * (d[c] - m) / s
    lp += np.array([P["bairro"].get(b, 0.0) for b in d["Neighborhood"]])
    lp += rng.normal(0, P["ruido"], n)
    d["Sale_Price"] = np.round(np.exp(lp)).astype(int)
    df = pd.DataFrame(d)[ESP["ordem"]]
    df.insert(0, "rownames", np.arange(1, n + 1))
    return df


print("Plano B pronto (so e usado se a base real nao carregar).")

In [ ]:
# Bloco 2 — as vendas de imoveis de Ames, Iowa (De Cock, 2011).
#
# Tres caminhos, nesta ordem:
#   1. o arquivo ames_housing.csv, na pasta Dados do repositorio;
#   2. o download do Rdatasets;
#   3. o pacote `rdatasets`, se estiver instalado.
# Se os tres falharem, segue o PLANO B (celula abaixo), com dados inventados.

ARQUIVO = "../Dados/ames_housing.csv"
URL = "https://vincentarelbundock.github.io/Rdatasets/csv/modeldata/ames.csv"


def carregar_ames(arquivo=ARQUIVO, url=URL, timeout=120):
    if os.path.exists(arquivo):
        return pd.read_csv(arquivo), "arquivo local " + arquivo
    try:
        import requests
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text)), "Rdatasets (download)"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    try:
        import rdatasets
        return rdatasets.data("modeldata", "ames"), "pacote rdatasets"
    except Exception:
        pass
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Mesmas colunas, mesmos")
    print("tipos e mesmas categorias da base real; precos e relacoes sao")
    print("sinteticos. Os numeros NAO valem como resultado sobre Ames.")
    print("=" * 70)
    return ames_sintetica(ESP_AMES), "PLANO B (sintetico)"


ames_bruta, FONTE = carregar_ames()
print(f"fonte: {FONTE}")
print(f"{ames_bruta.shape[0]} vendas, {ames_bruta.shape[1]} colunas")

In [ ]:
# Bloco 2 — o primeiro olhar. COMPLETE.
#
# Responda com codigo: quantas colunas sao numericas e quantas sao texto?
# Qual o preco mediano? Quantas vendas por ano? Que tipos de venda existem?

# COMPLETE
___

In [ ]:
# Bloco 2 — da tabela crua a matriz do modelo. COMPLETE.
def preparar_ames(dados):
    d = dados.drop(columns=[c for c in ("rownames", "Unnamed: 0") if c in dados.columns])
    # COMPLETE: fique so com as vendas normais e com area construida ate 4.000 pes2
    d = ___
    d = d.reset_index(drop=True)
    # COMPLETE: o alvo e o LOG do preco
    y = ___
    ano = d["Year_Sold"].to_numpy()
    d = d.drop(columns=["Sale_Price"])
    cat = list(d.select_dtypes(exclude="number").columns)
    d[cat] = d[cat].fillna("Nenhum")
    # COMPLETE: uma coluna 0/1 por categoria, menos a de referencia (pd.get_dummies)
    X = ___
    treino = ano <= 2009
    X = X.loc[:, X[treino].std() > 0]      # coluna constante no treino nao informa nada
    return X, y, treino


X, y, treino = preparar_ames(ames_bruta)
print(f"{X.shape[0]} vendas, {X.shape[1]} colunas no modelo")
print(f"treino (2006-2009): {treino.sum()} | teste (2010): {(~treino).sum()}")

## Bloco 3 — três modelos, dois tamanhos de amostra

In [ ]:
# Bloco 3 — as ferramentas. Leia, nao precisa alterar.
#
# E o que voce escreveu a mao no Bloco 1, embalado em funcoes: padronizar
# DENTRO do pipeline (o padronizador so ve o treino), validacao cruzada com
# dobras embaralhadas e fixas, e a regra de um erro-padrao.

ALPHAS_AMES = np.logspace(-4, -0.5, 60)
ALPHAS_RIDGE_AMES = np.logspace(-1, 4, 60)


def ols(X, y):
    return make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)

def lasso_cv(X, y, alphas, k=5, semente=0):
    """LassoCV com dobras embaralhadas e fixas, sobre colunas padronizadas."""
    return make_pipeline(
        StandardScaler(),
        LassoCV(alphas=alphas, cv=KFold(k, shuffle=True, random_state=semente),
                max_iter=50000)).fit(X, y)

def ridge_cv(X, y, alphas):
    return make_pipeline(StandardScaler(), RidgeCV(alphas=alphas)).fit(X, y)

def lambda_1ep(modelo_cv):
    """Regra de um erro-padrão: o maior λ cujo erro de validação cruzada fica
    a até um erro-padrão do mínimo. Mesmo erro, menos colunas."""
    L = modelo_cv[-1]
    media = L.mse_path_.mean(axis=1)
    ep = L.mse_path_.std(axis=1) / np.sqrt(L.mse_path_.shape[1])
    i = int(np.argmin(media))
    return float(L.alphas_[media <= media[i] + ep[i]].max())

def lasso(X, y, alpha):
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=50000)).fit(X, y)


print("Ferramentas prontas.")

In [ ]:
# Bloco 3 — OLS, Ridge e Lasso com todas as vendas de 2006-2009. COMPLETE.
Xtr, ytr = X[treino], y[treino]
Xte, yte = X[~treino], y[~treino]

# COMPLETE: ajuste os tres modelos no treino
modelos = {
    "OLS": ___,
    "Ridge": ___,
    "Lasso": ___,
}

# COMPLETE: RMSE em 2010 de cada um
erros = ___

for nome, e in erros.items():
    print(f"{nome:<6s} RMSE {e:.4f}  ->  erro tipico de cerca de {100 * (np.exp(e) - 1):.1f}% no preco")

In [ ]:
# Bloco 3 — o grafico da validacao cruzada. COMPLETE.
#
# O LassoCV guarda o erro de cada lambda em cada dobra em `mse_path_`
# (linhas = lambdas, colunas = dobras) e a grade em `alphas_`.
L = modelos["Lasso"][-1]
media = L.mse_path_.mean(axis=1)
ep = L.mse_path_.std(axis=1) / np.sqrt(L.mse_path_.shape[1])
a_min = L.alpha_
a_1ep = lambda_1ep(modelos["Lasso"])

# COMPLETE: desenhe a raiz do erro medio contra lambda (escala log), com a faixa
# de mais ou menos um erro-padrao, e marque a_min e a_1ep com linhas verticais
___

In [ ]:
# Bloco 3 — e com 300 vendas? COMPLETE.
rng = np.random.default_rng(200)
sub = rng.choice(np.flatnonzero(treino), 300, replace=False)
X300, y300 = X.iloc[sub], y[sub]

# COMPLETE: os mesmos tres modelos, treinados so nas 300 vendas, medidos em 2010
erros_300 = ___

tabela = pd.DataFrame({"todas as vendas": erros, "300 vendas": erros_300}).round(4)
print(tabela)

### ✔ CHECKPOINT I

Com todas as vendas, OLS e Lasso erram praticamente o mesmo em 2010. Com 300 vendas,
o OLS erra muito mais.

Em duas frases, usando as palavras **viés** e **variância**: por que a penalização quase
não faz diferença com 2.111 vendas e faz toda a diferença com 300?

## Bloco 4 — ler o que o Lasso escolheu

Com a penalização da regra de um erro-padrão: menos colunas, o mesmo erro.

In [ ]:
# Bloco 4 — as sobreviventes. COMPLETE.
lasso_1ep = lasso(Xtr, ytr, a_1ep)
coef = pd.Series(lasso_1ep[-1].coef_, index=X.columns)
desvio = pd.Series(lasso_1ep[0].scale_, index=X.columns)    # desvio-padrao de cada coluna no treino

# COMPLETE: quantas colunas ficaram? quais sao as 15 de maior |coeficiente|?
sobreviventes = ___
print(f"{len(sobreviventes)} de {len(coef)} colunas ficaram")
___

In [ ]:
# Bloco 4 — de volta a unidade original. COMPLETE.
#
# O coeficiente padronizado mede o efeito de UM DESVIO-PADRAO da coluna sobre o
# log do preco. Para uma variacao `delta` na unidade original:
#     efeito em % = 100 * (exp(coef / desvio * delta) - 1)

def efeito_pct(coluna, delta):
    # COMPLETE
    return ___


for coluna, delta, leitura in [
    ("Gr_Liv_Area", 107.639, "10 m2 a mais de area construida"),
    ("Fireplaces", 1, "uma lareira a mais"),
    ("Garage_Cars", 1, "uma vaga de garagem a mais"),
    ("Year_Built", 10, "construida 10 anos depois"),
    ("Bedroom_AbvGr", 1, "um quarto a mais, com a mesma area"),
]:
    print(f"{leitura:<38s} {efeito_pct(coluna, delta):+6.1f}%")

### Responda

O coeficiente do número de quartos é **negativo**. Mais quartos baixam o preço de uma casa?
Escreva a leitura correta em uma frase.

## Bloco 5 — a lista resiste a outra amostra?

O mesmo Lasso, com o **mesmo λ**, em 100 metades diferentes das vendas de 2006 a 2009.

In [ ]:
# Bloco 5 — 100 metades. COMPLETE.
idx = np.flatnonzero(treino)
rng = np.random.default_rng(7)
selecionada = []
for _ in range(100):
    metade = rng.choice(idx, len(idx) // 2, replace=False)
    # COMPLETE: rode o Lasso com lambda = a_1ep nesta metade e guarde quem ficou
    m = ___
    selecionada.append(___)

freq = pd.Series(np.mean(selecionada, axis=0), index=X.columns)

# COMPLETE: quantas entram SEMPRE (>= 99%), NUNCA (<= 1%) e AS VEZES?
___

In [ ]:
# Bloco 5 — os pares que se revezam. COMPLETE.
#
# Para cada par: a correlacao entre as duas colunas e a frequencia com que
# cada uma foi mantida.
pares = [("Gr_Liv_Area", "First_Flr_SF"), ("Gr_Liv_Area", "TotRms_AbvGrd"),
         ("Garage_Cars", "Garage_Area")]

# COMPLETE
___

### ✔ CHECKPOINT II

Um colega propõe publicar as colunas mantidas pelo Lasso como *“os determinantes do preço
dos imóveis em Ames”*.

Em duas frases: o que você responde a ele? Use o resultado do Bloco 5 e o que a Aula 03
mostrou sobre prever e explicar.

### Responda

> A prefeitura de Ames quer **estimar o valor venal** de todos os imóveis para cobrar o
> imposto predial. Qual dos seus modelos ela deveria usar? E se ela quisesse saber
> **quanto uma nova linha de ônibus valoriza** os imóveis do bairro, o mesmo modelo
> serviria?